# step 1 import everything we need 

In [41]:
import os 
from dotenv import load_dotenv 

# langchain 
from langchain_community.document_loaders import TextLoader 

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import JinaEmbeddings

In [42]:
load_dotenv()  # Load environment variables from .env file

True

In [43]:
groq_key = os.getenv("GROQ_KEY")
jina_key = os.getenv("JINA_KEY")

print("env variables loaded successfully.")

env variables loaded successfully.


# load our data

In [44]:
DATA_FILE_PATH = os.path.join("data", "dummy_hr_policy_data.txt")
print(f"Data file path: {DATA_FILE_PATH}")

Data file path: data\dummy_hr_policy_data.txt


# data ingetion part

In [45]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()

print("DATA LOADED SUCCESSFULLY")
print("="*40)
print(documents)

DATA LOADED SUCCESSFULLY
[Document(metadata={'source': 'data\\dummy_hr_policy_data.txt'}, page_content="DUMMY COMPANY HR POLICY DOCUMENT\nCompany: ExampleTech Solutions Pvt. Ltd.\nDocument Type: Sample HR Policy\nVersion: 1.0\nEffective Date: 01 January 2026\nStatus: Fictional sample for learning and RAG project practice\n\nIMPORTANT NOTICE\nThis document contains entirely fictional policies for demonstration and educational use.\nIt is not legal advice and does not represent any real employer's rules. Replace these\nexamples with approved company policies before using an HR chatbot in a real workplace.\n\n============================================================\n1. WORKING HOURS AND ATTENDANCE\n============================================================\n- Standard working days are Monday through Friday.\n- Standard working hours are 9:30 AM to 6:00 PM.\n- Employees receive a 60-minute lunch break.\n- Employees should record attendance using the company's attendance system.\n- Em

In [46]:
len(documents)

1

In [47]:
print(documents[0].page_content)

DUMMY COMPANY HR POLICY DOCUMENT
Company: ExampleTech Solutions Pvt. Ltd.
Document Type: Sample HR Policy
Version: 1.0
Effective Date: 01 January 2026
Status: Fictional sample for learning and RAG project practice

IMPORTANT NOTICE
This document contains entirely fictional policies for demonstration and educational use.
It is not legal advice and does not represent any real employer's rules. Replace these
examples with approved company policies before using an HR chatbot in a real workplace.

1. WORKING HOURS AND ATTENDANCE
- Standard working days are Monday through Friday.
- Standard working hours are 9:30 AM to 6:00 PM.
- Employees receive a 60-minute lunch break.
- Employees should record attendance using the company's attendance system.
- Employees who expect to arrive more than 15 minutes late should notify their manager.
- Flexible working arrangements require manager approval.
- Some teams may follow different schedules based on business needs.

Example question: What are the st

In [48]:
print(documents[0].metadata)

{'source': 'data\\dummy_hr_policy_data.txt'}


In [49]:
print(f"total characters of  document : {len(documents[0].page_content)}")

total characters of  document : 11064


# Splitiing our data 

In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50)

chunks= text_splitter.split_documents(documents)

print(chunks)


[Document(metadata={'source': 'data\\dummy_hr_policy_data.txt'}, page_content="DUMMY COMPANY HR POLICY DOCUMENT\nCompany: ExampleTech Solutions Pvt. Ltd.\nDocument Type: Sample HR Policy\nVersion: 1.0\nEffective Date: 01 January 2026\nStatus: Fictional sample for learning and RAG project practice\n\nIMPORTANT NOTICE\nThis document contains entirely fictional policies for demonstration and educational use.\nIt is not legal advice and does not represent any real employer's rules. Replace these\nexamples with approved company policies before using an HR chatbot in a real workplace."), Document(metadata={'source': 'data\\dummy_hr_policy_data.txt'}, page_content="============================================================\n1. WORKING HOURS AND ATTENDANCE\n============================================================\n- Standard working days are Monday through Friday.\n- Standard working hours are 9:30 AM to 6:00 PM.\n- Employees receive a 60-minute lunch break.\n- Employees should record at

In [51]:
len(chunks)

34

each splited chunk is a document - page content and metadata

In [52]:
print(chunks[0])

page_content='DUMMY COMPANY HR POLICY DOCUMENT
Company: ExampleTech Solutions Pvt. Ltd.
Document Type: Sample HR Policy
Version: 1.0
Effective Date: 01 January 2026
Status: Fictional sample for learning and RAG project practice

IMPORTANT NOTICE
This document contains entirely fictional policies for demonstration and educational use.
It is not legal advice and does not represent any real employer's rules. Replace these
examples with approved company policies before using an HR chatbot in a real workplace.' metadata={'source': 'data\\dummy_hr_policy_data.txt'}


In [53]:
print(chunks[5])

page_content='- Employees should request planned leave through the HR portal at least 3 working
  days in advance.
- Sick leave should be reported to the manager as soon as reasonably possible.
- Leave approval depends on team coverage and the applicable approval workflow.
- Unused leave carry-forward rules are subject to the company's annual leave procedure.
- Employees should contact HR for leave balances or questions about eligibility.' metadata={'source': 'data\\dummy_hr_policy_data.txt'}


In [54]:
print(chunks[7].page_content)

3. REMOTE WORK AND HYBRID WORK
- Employees may request remote work with their manager's approval.
- The sample hybrid arrangement is up to 3 remote working days per week.
- Team or client requirements may require employees to work from the office.
- Employees working remotely must protect company information and use approved devices.


In [55]:
print(chunks[8].page_content)

- Employees must remain reachable during their agreed working hours.
- Remote work from another country requires written approval from HR and Information
  Security before the employee makes arrangements.


In [56]:
print(chunks[5].page_content)

- Employees should request planned leave through the HR portal at least 3 working
  days in advance.
- Sick leave should be reported to the manager as soon as reasonably possible.
- Leave approval depends on team coverage and the applicable approval workflow.
- Unused leave carry-forward rules are subject to the company's annual leave procedure.
- Employees should contact HR for leave balances or questions about eligibility.


## embedd our data

In [57]:
from langchain_community.embeddings import JinaEmbeddings

embedding_model=JinaEmbeddings(model_name="jina-embeddings-v4")

print("EMB READY THE NAME IS ",embedding_model.model_name)

EMB READY THE NAME IS  jina-embeddings-v4


## STORE DATA IN VECTORE DB

In [58]:
from langchain_community.vectorstores import FAISS

vector_store=FAISS.from_documents(chunks, embedding_model)
print("chunks are stored", vector_store.index.ntotal)


chunks are stored 34


In [59]:
test_query="how many sick leave employess get"

## similarity search

top_matches=vector_store.similarity_search(test_query, k=2)
print(f"Query :{test_query}\n")
for i, match in enumerate(top_matches, start=1):
    print(f"---match{i}---")
    print(match.page_content)
    print()      

Query :how many sick leave employess get

---match1---
2. LEAVE POLICY
- Full-time employees receive 12 days of casual leave per calendar year.
- Full-time employees receive 12 days of sick leave per calendar year.
- Full-time employees receive 18 days of earned leave per calendar year.
- Leave entitlements are prorated for eligible employees who join during the year.

---match2---
Example question: How many earned leave days are provided?
Sample answer: This fictional policy provides eligible full-time employees with
18 days of earned leave per calendar year.



### Tool

In [60]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


def search_hr_policy(question: str) -> str:
    """Search the HR policy knowledge base for company 
       guidance on topics such as leave, benefits, and
       workplace policies. Use the search results to 
       answer HR questions; do not guess if the information
       is not found."""
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

## Data retrival 

### LLM

In [61]:
from langchain_groq import ChatGroq
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5
)
llm.model_name

'openai/gpt-oss-120b'

In [62]:
test_response=llm.invoke("hey is learning rag is hard ? reply in a joke manner , in one line")

In [63]:
test_response.content

'Learning RAG is like juggling cats—hard at first, but once you get the hang of it, you’ll be purring with success!'

### AI AGENT

3--

LLM - BRAIN

TOOL - SUPER POWER

MEMORY - NO MEMORY 


In [64]:
from langchain.agents import create_agent


In [65]:
hr_assistant=create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
    you are a friendly hr assistant.
    always use the search_hr_policy tool to look up
    facts before answering.
    if the answer isn't in the search results, say you dont know"
    instead of guessing."
    """

)
print("hr assistant agent is ready to answer questions. ")

hr assistant agent is ready to answer questions. 


In [66]:
def ask_hr_assistant(question: str) -> str:
    """send a question to the rag agent and print a nicely formmated answer."""
    print("="*60)
    print("Question:",question) 
    print("-"*60)

    response = hr_assistant.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    answer = response["messages"][-1].content
    answer= response["message"][-1]["content"]
    print("Answer:",answer)
    print("="*60)
    print()
    return answer

In [74]:
response= hr_assistant.invoke({
    "messages": [{"role": "user",
                   "content": "tell me which organisation you work for ?"}]
})


In [75]:
response

{'messages': [HumanMessage(content='tell me which organisation you work for ?', additional_kwargs={}, response_metadata={}, id='c4fab6f5-333b-482c-a93b-ebc5769dd2fe'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "tell me which organisation you work for ?" As HR assistant, need to answer based on policy. Probably we have no policy about disclosing organization. Might be internal policy: maybe we are a generic HR assistant for "Acme Corp". But we need to use search_hr_policy to look up. Let\'s search.', 'tool_calls': [{'id': 'fc_ebab990d-13a2-4476-a024-d1e9064faab3', 'function': {'arguments': '{"question":"organization name of HR assistant"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 219, 'total_tokens': 319, 'completion_time': 0.20743476, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.009138817, 'prompt_tokens_details': None, 'queue_time':

system message - hr assistant

human message - tell me about polices

ai message - hey these are the polices

In [76]:
response["messages"][1].content

''

In [70]:
response["messages"][-2].content

"Example question: How many earned leave days are provided?\nSample answer: This fictional policy provides eligible full-time employees with\n18 days of earned leave per calendar year.\n\n============================================================\n2. LEAVE POLICY\n============================================================\n- Full-time employees receive 12 days of casual leave per calendar year.\n- Full-time employees receive 12 days of sick leave per calendar year.\n- Full-time employees receive 18 days of earned leave per calendar year.\n- Leave entitlements are prorated for eligible employees who join during the year.\n\n- Employees should request planned leave through the HR portal at least 3 working\n  days in advance.\n- Sick leave should be reported to the manager as soon as reasonably possible.\n- Leave approval depends on team coverage and the applicable approval workflow.\n- Unused leave carry-forward rules are subject to the company's annual leave procedure.\n- Employees 

In [71]:
response["messages"][-3].content

''

In [72]:
response["messages"][-4].content

'tell me about leave policies how to apply for leave and how many leaves are allowed per year'

In [77]:
print(response["messages"][-1].content)

I’m the HR assistant for **ExampleTech Solutions Pvt. Ltd.**.
